# Part 2 — MapReduce
**Input** : `crash_manhattan_alcohol_clean.csv`  
**Output** : `crash_zones_mr_ready.csv`

Schema: `zone_id, crash_date, crash_hour, crash_count, injury_count, death_count`

Steps:
1. Load clean crash data
2. MAP — assign H3 zone_id (res 10) + parse crash_hour per row
3. REDUCE — aggregate counts per (zone_id, crash_date, crash_hour)
4. Save

In [9]:
import h3
import pandas as pd
import numpy as np
from collections import defaultdict

INPUT  = "crash_manhattan_alcohol_clean.csv"
OUTPUT = "crash_zones_mr_ready.csv"
H3_RES = 10

df = pd.read_csv(INPUT, dtype=str, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["number_of_persons_injured"] = pd.to_numeric(df.get("number_of_persons_injured"), errors="coerce").fillna(0)
df["number_of_persons_killed"]  = pd.to_numeric(df.get("number_of_persons_killed"),  errors="coerce").fillna(0)

df = df.dropna(subset=["latitude", "longitude"])
print(f"Loaded {len(df):,} rows")

Loaded 2,907 rows


In [10]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# Each row → key=(zone_id, crash_date, crash_hour), val=(crashes, injured, killed)

def parse_hour(t):
    try: return int(str(t).split(":")[0])
    except: return 0

def mapper(row):
    try:
        zone_id = h3.latlng_to_cell(float(row["latitude"]), float(row["longitude"]), H3_RES)
    except:
        return None, None
    key = (
        zone_id,
        str(pd.to_datetime(row.get("crash_date"), errors="coerce"))[:10],
        parse_hour(row.get("crash_time", "0:00")),
    )
    val = {
        "crashes": 1,
        "injured": float(row["number_of_persons_injured"]),
        "killed":  float(row["number_of_persons_killed"]),
    }
    return key, val

mapped = [mapper(row) for _, row in df.iterrows()]
mapped = [(k, v) for k, v in mapped if k is not None]
print(f"Mapped pairs: {len(mapped):,}")

Mapped pairs: 2,907


In [11]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
# Sum crash_count, injury_count, death_count per (zone_id, crash_date, crash_hour)

acc = defaultdict(lambda: {"crashes":0, "injured":0.0, "killed":0.0})

for key, val in mapped:
    a = acc[key]
    a["crashes"] += val["crashes"]
    a["injured"] += val["injured"]
    a["killed"]  += val["killed"]

rows = []
for (zone_id, crash_date, crash_hour), a in acc.items():
    rows.append({
        "zone_id":     zone_id,
        "crash_date":  crash_date,
        "crash_hour":  crash_hour,
        "crash_count": a["crashes"],
        "injury_count":a["injured"],
        "death_count": a["killed"],
    })

out = pd.DataFrame(rows).sort_values(["zone_id","crash_date","crash_hour"]).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f"Output rows  : {len(out):,}")
print(f"Unique zones : {out['zone_id'].nunique():,}")
print(f"Saved → {OUTPUT}")
out.head(10)

Output rows  : 2,893
Unique zones : 1,490
Saved → crash_zones_mr_ready.csv


,zone_id,crash_date,crash_hour,crash_count,injury_count,death_count
0,8a2a1008800ffff,2017-02-11,1,1,2.0,0.0
1,8a2a1008800ffff,2019-02-15,20,1,1.0,0.0
2,8a2a1008802ffff,2020-11-05,0,1,0.0,0.0
3,8a2a1008802ffff,2022-10-28,23,1,1.0,0.0
4,8a2a10088047fff,2015-04-01,17,1,0.0,0.0
5,8a2a10088047fff,2020-10-06,20,1,0.0,0.0
6,8a2a10088047fff,2023-10-21,3,1,0.0,0.0
7,8a2a1008804ffff,2015-11-14,6,1,0.0,0.0
8,8a2a1008804ffff,2018-12-04,21,1,1.0,0.0
9,8a2a100880e7fff,2013-11-25,18,1,0.0,0.0
